# Passo 1: Extração (SQL Server para MinIO - CSV)
Neste notebook, conectamos no SQL Server, lemos as tabelas e gravamos no bucket `landing-zone` no formato CSV.

In [ ]:
from pyspark.sql import SparkSession

# Inicializa o Spark com pacotes do SQL Server e MinIO (AWS S3)
spark = SparkSession.builder \
    .appName("Extract-To-Landing") \
    .config("spark.jars.packages", "com.microsoft.sqlserver:mssql-jdbc:12.6.1.jre11,org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:9000") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

jdbc_url = "jdbc:sqlserver://localhost:1433;databaseName=master;encrypt=false;trustServerCertificate=true"
jdbc_props = {
    "user": "sa",
    "password": "SqlServer@2024!",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

# Lendo a tabela do SQL Server
df_clientes = spark.read.jdbc(url=jdbc_url, table="clientes", properties=jdbc_props)
df_clientes.show()

# Escrevendo no MinIO (bucket landing-zone) em formato CSV
landing_path = "s3a://landing-zone/clientes_csv"
df_clientes.write.csv(landing_path, header=True, mode="overwrite")

print("Dados exportados para o MinIO em CSV com sucesso!")
